# Stage 2 -- zero-shot LLM applicability feature (H24 step D)

**What this produces.** For every Stage 2 row (690 train + 173 test), the
probability that an instruction-tuned open LLM answers **1 (applicable)**
rather than **0 (not applicable)**. It is read from a single forward pass: the
next-token logits of `1` and `0`. There is no sampling, so the output is
deterministic up to floating-point noise.

**Why it is fold-safe.** The prompt is zero-shot, so no training label enters
any prompt. The score is a row-local feature, like age or the ICD code, and it
can be used inside cross-validation without leaking.

**Runtime.** A Colab T4 is enough; the model loads in 4-bit. Expect roughly
20-40 minutes for 863 rows. Each row is cached to Google Drive, so a
disconnect only loses the row in progress.

**Afterwards.** Copy `llm_feature_train.csv`, `llm_feature_test.csv` and
`llm_manifest.json` into the project's `h24/llm/` folder, then run:

```
python src/stage2_h24_step_d_eval.py
```

This notebook is generated by `src/build_llm_colab.py`; edit that script, not
this file.

In [ ]:
# ---- 1. dependencies (torch is preinstalled on Colab) ----
!pip -q install "transformers>=4.45" accelerate bitsandbytes pandas scikit-learn

In [ ]:
# ---- 2. configuration ----
import hashlib, json, math, os, random, time
from pathlib import Path

import numpy as np
import pandas as pd
import torch

SEED = 42
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"   # strong multilingual/Russian open model
MODEL_REVISION = "main"                  # after the first run, pin the sha recorded in llm_manifest.json
LOAD_IN_4BIT = None                      # None = auto (4-bit when the GPU has < 20 GB)
MAX_PROTOCOL_CHARS = 7000                # cleaned protocols are <= 6,794 chars, so nothing is cut
USE_DRIVE = True

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
assert torch.cuda.is_available(), "Runtime -> Change runtime type -> GPU"

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = Path("/content/drive/MyDrive/aiijc_llm")
else:
    WORK = Path("/content/aiijc_llm")
CACHE = WORK / "cache"
CACHE.mkdir(parents=True, exist_ok=True)
print("working dir:", WORK, "| GPU:", torch.cuda.get_device_name(0))

In [ ]:
# ---- 3. data: train_stage2.csv and test_stage2.csv ----
needed = ["train_stage2.csv", "test_stage2.csv"]
missing = [f for f in needed if not (WORK / f).exists()]
if missing:
    from google.colab import files
    print("upload:", missing)
    for name, data in files.upload().items():
        (WORK / name).write_bytes(data)
train = pd.read_csv(WORK / "train_stage2.csv", encoding="utf-8")
test = pd.read_csv(WORK / "test_stage2.csv", encoding="utf-8")
assert train.shape == (690, 4) and test.shape == (173, 3), (train.shape, test.shape)
print(train.shape, test.shape)

In [ ]:
# ---- 4. text preparation, copied verbatim from src/stage2_h24_features.py ----
import re

ICD_PATTERN = re.compile('Код МКБ-10\\s*:\\s*([A-Z]\\d{2}(?:\\.\\d{1,2})?)')
ICD_IN_DIAGNOSIS = re.compile('Диагноз\\s*:\\s*([A-Z]\\d{2}(?:\\.\\d{1,2})?)')
DIAGNOSIS_FIELDS = ('Диагноз', 'Основное заболевание', 'Заключение', 'Сопутствующие заболевания', 'Осложнения')
LAB_LINE = re.compile('Показатель:[^\\n]*')
BLANK_RUNS = re.compile('\\s*\\n\\s*(?:\\n\\s*)+')
MISSING = 'NA'


def icd_code(text: str) -> str:
    """Full ICD-10 code (e.g. ``K50.1``), or ``NA`` when the protocol has none."""
    text = str(text)
    m = ICD_PATTERN.search(text) or ICD_IN_DIAGNOSIS.search(text)
    return m.group(1) if m else MISSING


def diagnosis_text(text: str) -> str:
    """The diagnosis-bearing fields, joined; empty when the protocol has none."""
    text = str(text)
    chunks = []
    for field in DIAGNOSIS_FIELDS:
        for m in re.finditer(rf"{re.escape(field)}\s*:\s*([^\n]+)", text):
            chunks.append(m.group(1).strip())
    return " ".join(chunks)


def clean_protocol(text: str) -> str:
    """Full protocol minus the lab-value boilerplate lines."""
    return BLANK_RUNS.sub("\n", LAB_LINE.sub("", str(text))).strip()


In [ ]:
# ---- 5. prompt ----
SYSTEM = ("Ты — врач-эксперт. Ты проверяешь, применим ли раздел клинических рекомендаций "
          "к конкретному пациенту, по протоколу его осмотра.")

INSTRUCTION = """Задача: определить, применим ли этот раздел к данному пациенту.
Раздел описывает особую группу пациентов (возраст, пол, беременность, сопутствующее заболевание, форма, локализация или тяжесть болезни).
Ответ 1 (применимо): пациент относится к этой группе, либо данных недостаточно, чтобы это исключить, и принадлежность к группе правдоподобна.
Ответ 0 (не применимо): протокол противоречит условию раздела — например, другой возраст или пол, другая локализация, форма или тяжесть болезни по диагнозу или коду МКБ-10, либо принадлежность к группе явно неправдоподобна по имеющимся данным (например, постменопауза у пациентки репродуктивного возраста).
Отрицания («не выявлено», «отрицает», «без», «нет») означают, что признак отсутствует.
Ответь одной цифрой: 1 или 0."""


def _field(text, name):
    m = re.search(rf"{re.escape(name)}\s*:\s*([^\n]+)", text)
    return m.group(1).strip() if m else "не указано"


def build_prompt(title, protocol):
    protocol = str(protocol)
    code = icd_code(protocol)
    card = "\n".join([
        f"Пол: {_field(protocol, 'Пол')}",
        f"Возраст: {_field(protocol, 'Возраст')}",
        f"Код МКБ-10: {code if code != MISSING else 'не указан'}",
        f"Диагноз: {diagnosis_text(protocol) or 'не указан'}",
    ])
    user = (f"Раздел клинических рекомендаций (путь в документе):\n{str(title).strip()}\n\n"
            f"Протокол осмотра пациента (ключевые поля):\n{card}\n\n"
            f"Полный протокол (без таблиц лабораторных показателей):\n"
            f"{clean_protocol(protocol)[:MAX_PROTOCOL_CHARS]}\n\n{INSTRUCTION}")
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}]


PROMPT_SHA = hashlib.sha256((SYSTEM + INSTRUCTION + str(MAX_PROTOCOL_CHARS)).encode()).hexdigest()
print("prompt sha:", PROMPT_SHA[:16])
print(build_prompt(train.title_text[0], train.protocol_text[0])[1]["content"][:1500])

In [ ]:
# ---- 6. model ----
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

gpu_gb = torch.cuda.get_device_properties(0).total_memory / 2**30
use_4bit = (gpu_gb < 20) if LOAD_IN_4BIT is None else LOAD_IN_4BIT
tok = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
kwargs = dict(revision=MODEL_REVISION, device_map="auto")
if use_4bit:
    kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
else:
    kwargs["torch_dtype"] = torch.bfloat16
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **kwargs).eval()

ID1, ID0 = tok.convert_tokens_to_ids("1"), tok.convert_tokens_to_ids("0")
assert tok.encode("1", add_special_tokens=False) == [ID1]
assert tok.encode("0", add_special_tokens=False) == [ID0]

try:
    from huggingface_hub import HfApi
    MODEL_SHA = HfApi().model_info(MODEL_ID, revision=MODEL_REVISION).sha
except Exception as exc:          # offline or rate-limited: fall back to the config
    MODEL_SHA = getattr(model.config, "_commit_hash", None) or f"unknown ({exc})"
print(f"GPU {gpu_gb:.0f} GB | 4-bit={use_4bit} | model sha {MODEL_SHA}")

In [ ]:
# ---- 7. scoring, cached per row ----
from tqdm.auto import tqdm


@torch.inference_mode()
def score_row(title, protocol):
    enc = tok.apply_chat_template(build_prompt(title, protocol), add_generation_prompt=True,
                                  tokenize=True, return_dict=True, return_tensors="pt")
    enc = {k: v.to(model.device) for k, v in enc.items()}
    logits = model(**enc).logits[0, -1].float()
    top = int(logits.argmax())
    return float(logits[ID1] - logits[ID0]), top in (ID1, ID0), int(enc["input_ids"].shape[1])


def score_split(frame, split):
    (CACHE / split).mkdir(exist_ok=True)
    rows, t0 = [], time.time()
    for rid, title, protocol in tqdm(list(zip(frame.id, frame.title_text, frame.protocol_text)), desc=split):
        path = CACHE / split / f"{rid}.json"
        rec = json.loads(path.read_text()) if path.exists() else None
        if not rec or rec.get("prompt_sha") != PROMPT_SHA or rec.get("model_sha") != MODEL_SHA:
            logit, on_format, n_tok = score_row(title, protocol)
            rec = {"id": int(rid), "logit_llm": logit, "answer_on_format": on_format,
                   "n_tokens": n_tok, "prompt_sha": PROMPT_SHA, "model_sha": MODEL_SHA}
            path.write_text(json.dumps(rec))
        rows.append(rec)
    out = pd.DataFrame(rows)
    out["p_llm"] = 1 / (1 + np.exp(-out["logit_llm"]))
    assert (out["id"].to_numpy() == frame["id"].to_numpy()).all()
    print(f"{split}: {len(out)} rows in {time.time() - t0:.0f}s | "
          f"top token was 0/1 for {out.answer_on_format.mean():.1%} | "
          f"max prompt {out.n_tokens.max()} tokens")
    return out


# quick smoke test before the long run
print(score_row(train.title_text[0], train.protocol_text[0]))

In [ ]:
# ---- 8. run ----
feat_train = score_split(train, "train")
feat_test = score_split(test, "test")
cols = ["id", "p_llm", "logit_llm"]
feat_train[cols].to_csv(WORK / "llm_feature_train.csv", index=False)
feat_test[cols].to_csv(WORK / "llm_feature_test.csv", index=False)

manifest = {
    "model_id": MODEL_ID, "model_revision_requested": MODEL_REVISION, "model_sha": MODEL_SHA,
    "load_in_4bit": use_4bit, "gpu": torch.cuda.get_device_name(0),
    "prompt_sha256": PROMPT_SHA, "system_prompt": SYSTEM, "instruction": INSTRUCTION,
    "max_protocol_chars": MAX_PROTOCOL_CHARS, "seed": SEED,
    "versions": {"torch": torch.__version__, "transformers": transformers.__version__},
    "answer_on_format_rate": {"train": float(feat_train.answer_on_format.mean()),
                              "test": float(feat_test.answer_on_format.mean())},
    "labels_used": "none (zero-shot); train labels are used below for reporting only",
}
(WORK / "llm_manifest.json").write_text(json.dumps(manifest, indent=2, ensure_ascii=False))
print(json.dumps(manifest, indent=1, ensure_ascii=False)[:800])

In [ ]:
# ---- 9. report only: how informative is the score? (nothing is tuned on this) ----
from sklearn.metrics import fbeta_score, roc_auc_score

y = train["label"].to_numpy()
p = feat_train["p_llm"].to_numpy()
print("train AUC: %.4f" % roc_auc_score(y, p))
for t in (0.3, 0.5, 0.7):
    f2 = fbeta_score(y, (p >= t).astype(int), beta=2, average="macro")
    print("threshold %.1f  macro-F2 %.4f  M2 %.4f" % (t, f2, max(0.0, min(1.0, (f2 - 0.5) / 0.45))))
last = train.title_text.str.strip().str.split("\n").str[-1]
per_title = []
for t_, g in pd.DataFrame({"t": last, "y": y, "p": p}).groupby("t"):
    if g.y.nunique() > 1:
        per_title.append(roc_auc_score(g.y, g.p))
print("mean within-title AUC: %.4f over %d titles" % (np.mean(per_title), len(per_title)))

In [ ]:
# ---- 10. download the three files for h24/llm/ ----
from google.colab import files
for name in ("llm_feature_train.csv", "llm_feature_test.csv", "llm_manifest.json"):
    files.download(str(WORK / name))